# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded, get_data_compacted

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
1291,10,62,"(0.1757680747561528, 0.7069953711641571)","(0.004, 0.786, 0.046, -0.777)",1,1.0,"(0.02, 0.978, 0.03, -0.985)",1.0,1.0,"(0.039, 1.169, 0.011, -1.196)",...,0.046,-0.777,0.020,0.978,0.030,-0.985,0.039,1.169,0.011,-1.196
955,26,47,"(0.1677354969823425, 0.9471294840196408)","(-0.112, 0.393, 0.131, -0.122)",1,1.0,"(-0.104, 0.581, 0.129, -0.259)",0.0,1.0,"(-0.092, 0.391, 0.124, -0.076)",...,0.131,-0.122,-0.104,0.581,0.129,-0.259,-0.092,0.391,0.124,-0.076
1285,4,62,"(0.1757680747561528, 0.7069953711641571)","(-0.01, -0.36, 0.053, 0.447)",1,1.0,"(-0.017, -0.169, 0.062, 0.241)",1.0,1.0,"(-0.021, 0.022, 0.067, 0.037)",...,0.053,0.447,-0.017,-0.169,0.062,0.241,-0.021,0.022,0.067,0.037
602,17,31,"(0.4456006051678612, 0.2809654390334451)","(0.049, 0.159, -0.018, -0.449)",1,1.0,"(0.052, 0.373, -0.027, -1.16)",0.0,1.0,"(0.06, 0.16, -0.051, -0.484)",...,-0.018,-0.449,0.052,0.373,-0.027,-1.160,0.060,0.160,-0.051,-0.484
1282,1,62,"(0.1757680747561528, 0.7069953711641571)","(0.011, -0.167, 0.028, 0.204)",0,1.0,"(0.008, -0.359, 0.032, 0.43)",0.0,1.0,"(0.001, -0.551, 0.04, 0.656)",...,0.028,0.204,0.008,-0.359,0.032,0.430,0.001,-0.551,0.040,0.656


# Predict 

In [4]:
def optim_params(prediction_dataset):
    weights = prediction_dataset.copy()
    for m, _ in enumerate(models):
        weights['estimated_s'] = prediction_dataset[f'estimated_s_model_{m}']
        expansions = {f'estimated_s': [f'estimated_s0', f'estimated_s1', f'estimated_s2', f'estimated_s3']}
        ds = get_data_expanded(weights, expansions)
        # weights[f'estimated_s_model_{m}'] = ds['estimated_s']
        for d in range(4):
            weights[f'estimated_s{d}_model{m}'] = ds[f'estimated_s{d}']
    weights['A'] = weights.apply(lambda row: np.array([[row[f'estimated_s{d}_model{m}'] for m,_ in enumerate(models)] for d in range(4)]),axis=1)
    weights['b'] = weights.apply(lambda row: np.array([row[f's__{d}'] for d in range(4)]),axis=1)
    weights['w'] = weights.apply(lambda row: np.linalg.lstsq(row.A, row.b)[0],axis=1)

    weights[[f'estimated_weight_{m}' for m, _ in enumerate(models)]] = weights.apply(lambda row: pd.Series(row['w']),axis=1)
    return weights


In [5]:

def predict_with_params(prediction_dataset, weights):
    for m, _ in enumerate(models):
        prediction_dataset[f'weight_{m}'] = weights[f'estimated_weight_{m}']

    expansions = {
        f'estimated_s_model_{m}': [f'estimated_s{d}_model{m}' for d in range(4)] for m,_ in enumerate(models)}
    ds = get_data_expanded(prediction_dataset, expansions)
    
    for d in range(4):
        ds[f'estimated_weighted_s{d}'] = ds.apply(
            lambda row: np.sum([row[f'estimated_s{d}_model{m}']* row[f'weight_{m}'] for m, _ in enumerate(models)])
            , axis=1
        )
    
    ds = get_data_compacted(ds, {'estimated_weighted_s': [f'estimated_weighted_s{d}' for d in range(4)]})
     
    prediction_dataset['estimated_r'] = ds['estimated_r_model_0']
    prediction_dataset['estimated_s'] = ds['estimated_weighted_s']
    
    results = data.get_evaluation_metrics(prediction_dataset, p=False)
    prediction_dataset[f'rse'] = results['rse']
    prediction_dataset[f'rse_normalized'] = results['rse_normalized']

    prediction_dataset[f'rse_s0'] = results['rse_s0']
    prediction_dataset[f'rse_s1'] = results['rse_s1']
    prediction_dataset[f'rse_s2'] = results['rse_s2']
    prediction_dataset[f'rse_s3'] = results['rse_s3']

    prediction_dataset[f'rse_s0_normalized'] = results['rse_s0_normalized']
    prediction_dataset[f'rse_s1_normalized'] = results['rse_s1_normalized']
    prediction_dataset[f'rse_s2_normalized'] = results['rse_s2_normalized']
    prediction_dataset[f'rse_s3_normalized'] = results['rse_s3_normalized']

    return prediction_dataset

In [6]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_r_model_{i}'] = pred['estimated_r']
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [7]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_', 's_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3', 's__']] = pre_df[['s', 's0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3', 's_']]

    prediction_dataset = evaluate(models, pre_df)
    params = optim_params(prediction_dataset)
    prediction_dataset = evaluate(models, df)
    final_predictions = predict_with_params(prediction_dataset, params)

    cols = [
        's__', 'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ] + [f'weight_{m}' for m, _ in enumerate(models)] + [f'estimated_s_model_{m}' for m, _ in enumerate(models)]

    return final_predictions[cols]

In [8]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

,s__,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,...,weight_0,weight_1,weight_2,weight_3,weight_4,estimated_s_model_0,estimated_s_model_1,estimated_s_model_2,estimated_s_model_3,estimated_s_model_4
1291,"(0.039, 1.169, 0.011, -1.196)","(0.038954160283034805, 1.158300032316632, 0.00...",0.046425,0.505225,0.000046,0.010700,0.003160,0.032519,0.545983,0.528687,...,-0.186201,0.131147,0.468149,0.427631,0.196145,"(0.005, 1.116, 0.012, -1.188)","(0.008, 0.842, -0.004, 0.949)","(0.034, 1.141, 0.014, -1.303)","(0.038, 1.146, 0.009, -1.691)","(0.034, 1.18, 0.001, -1.229)"
955,"(-0.092, 0.391, 0.124, -0.076)","(-0.08930148552624176, 0.3956182604327235, 0.1...",0.040469,0.505577,0.002699,0.004618,0.003298,0.029855,0.548784,0.527192,...,0.076937,0.059378,0.437147,-0.102256,0.548775,"(-0.104, 0.377, 0.128, 0.237)","(-0.04, 0.374, 0.08, -0.047)","(-0.088, 0.383, 0.131, 0.088)","(-0.082, 0.373, 0.118, 0.576)","(-0.089, 0.392, 0.123, -0.075)"
1285,"(-0.021, 0.022, 0.067, 0.037)","(-0.021511575467712856, 0.027274213053250656, ...",0.025313,0.503418,0.000512,0.005274,0.000996,0.018531,0.546475,0.527354,...,0.141611,-0.053094,0.596921,-0.023239,0.363204,"(-0.018, 0.032, 0.068, -0.197)","(-0.018, 0.071, 0.079, -1.036)","(-0.023, 0.033, 0.066, -0.077)","(-0.015, 0.035, 0.066, -0.262)","(-0.018, 0.021, 0.068, 0.086)"
602,"(0.06, 0.16, -0.051, -0.484)","(0.04561053825459142, 0.1706754984177931, -0.0...",0.086759,0.517952,0.014389,0.010675,0.017663,0.044030,0.561129,0.528681,...,-0.086776,0.164208,0.866248,1.065563,-1.006602,"(0.057, 0.168, -0.054, -0.683)","(0.056, 0.163, -0.079, -0.643)","(0.06, 0.153, -0.057, -0.808)","(0.059, 0.167, -0.05, -0.55)","(0.073, 0.151, -0.042, -0.799)"
1282,"(0.001, -0.551, 0.04, 0.656)","(0.0022952473411551695, -0.5524975433991846, 0...",0.042330,0.503900,0.001295,0.001498,0.001132,0.038405,0.547302,0.526425,...,0.130503,-0.052418,0.237433,0.031749,0.650072,"(0.003, -0.562, 0.041, 0.903)","(-0.033, -0.558, 0.037, 0.908)","(0.001, -0.567, 0.041, 0.787)","(-0.002, -0.532, 0.042, 0.956)","(0.0, -0.549, 0.041, 0.626)"


In [9]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,weight_0,weight_1,weight_2,weight_3,weight_4
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.125805,0.509267,0.005800,0.015132,0.004492,0.100380,0.552059,0.529777,0.511972,0.443260,0.188047,0.037946,0.264301,0.121220,0.386456
std,0.243067,0.018896,0.028061,0.040982,0.014202,0.195663,0.029631,0.010074,0.034058,0.016742,0.950655,0.252498,0.788201,0.745243,0.712365
min,0.002115,0.502242,0.000002,0.000006,0.000001,0.000116,0.545936,0.526059,0.501202,0.434681,-8.863365,-2.398180,-7.651310,-12.339402,-8.471882
25%,0.024473,0.503769,0.000785,0.002419,0.000753,0.015316,0.546763,0.526652,0.503005,0.435982,-0.115257,-0.024394,0.047250,-0.103877,0.096317
50%,0.053069,0.505086,0.001928,0.006032,0.001704,0.039223,0.547970,0.527540,0.505286,0.438027,0.168004,0.011319,0.306255,0.060855,0.407142
75%,0.115390,0.508343,0.004146,0.014045,0.003591,0.091657,0.550312,0.529510,0.509810,0.442514,0.422545,0.079667,0.530281,0.333113,0.697026
max,2.834444,0.916785,0.875232,0.875086,0.287573,2.589052,1.470150,0.741172,1.190821,0.656204,14.979306,5.200842,7.079216,6.048611,6.254568
